<a href="https://colab.research.google.com/github/KamilAkarsu/DoktoraDeneyleri/blob/main/Exp_2_1_MURA_and_Traditional_CNN(resnet50).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
#Hücre 1: Kütüphane İçe Aktarımları ve Veri Ortamının Hazırlanması

# ==============================================================================
# TEMEL KÜTÜPHANELERİN İÇE AKTARILMASI
# ==============================================================================
import os           # İşletim sistemi işlemleri, klasör/dosya yolu yönetimi
import time         # Eğitim sürelerinin ve çıkarım (inference) hızlarının hesaplanması
import zipfile      # Sıkıştırılmış veri setinin (MURA) yerel diske çıkartılması

# ==============================================================================
# PYTORCH VE DERİN ÖĞRENME MODÜLLERİ
# ==============================================================================
import torch        # Ana derin öğrenme çerçevesi
import torch.nn as nn # Sinir ağı katmanları ve kayıp (loss) fonksiyonları
import torch.optim as optim # Optimizasyon algoritmaları (AdamW vb. ve Öğrenme Oranı Planlayıcıları)
from torch.utils.data import Dataset, DataLoader # Özelleştirilmiş veri seti ve toplu veri yükleyici sınıfları
from torchvision import transforms, models # Dinamik veri artırımı (augmentation) ve ön-eğitimli jenerik modeller

# ==============================================================================
# VERİ İŞLEME VE GÖRSELLEŞTİRME MODÜLLERİ
# ==============================================================================
from PIL import Image # Tıbbi görüntülerin (Röntgen) diskten okunması
import numpy as np  # Vektörel matris işlemleri
import pandas as pd # Eğitim geçmişinin ve sonuçların tablo halinde kaydedilmesi (CSV)
from tqdm import tqdm # Eğitim ve doğrulama döngülerinde iterasyon ilerleyişinin görselleştirilmesi

# ==============================================================================
# PERFORMANS METRİKLERİ (SCIKIT-LEARN)
# ==============================================================================
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                             f1_score, cohen_kappa_score, confusion_matrix,
                             roc_auc_score, average_precision_score,
                             mean_absolute_error, mean_squared_error)

# ==============================================================================
# ÇALIŞMA ORTAMI VE VERİ BAZI HAZIRLIKLARI
# ==============================================================================
# 1. Google Drive Bağlantısı: Ağırlıkların ve grafiklerin kaydedileceği kalıcı disk
from google.colab import drive
drive.mount('/content/drive')

# 2. Veri Setinin Yerel Diske Çıkartılması
# Bulut depolamadan (Drive) doğrudan veri okumak I/O darboğazına (bottleneck) yol açar.
# GPU'nun veriyi beklemesini engellemek için veri seti geçici, hızlı yerel belleğe alınır.
DRIVE_ZIP_YOLU = '/content/drive/MyDrive/Doktora/Verisetleri_tik/Islenmis_MURA.zip'
YEREL_VERI_KLASORU = '/content/dataset'

if not os.path.exists(YEREL_VERI_KLASORU):
    print(f"Veri seti yerel diske çıkartılıyor: {DRIVE_ZIP_YOLU} ...")
    with zipfile.ZipFile(DRIVE_ZIP_YOLU, 'r') as zip_ref:
        zip_ref.extractall(YEREL_VERI_KLASORU)
    print("Çıkartma işlemi tamamlandı.")
else:
    print("Veri seti yerel diskte zaten mevcut.")

# ==============================================================================
# DONANIM HIZLANDIRICI ATAMASI
# ==============================================================================
# A100 GPU Kontrolü: Tensor işlemlerinin CPU yerine CUDA mimarisinde yürütülmesi için
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Aktif İşlem Birimi: {device}")

Mounted at /content/drive
Veri seti yerel diske çıkartılıyor: /content/drive/MyDrive/Doktora/Verisetleri_tik/Islenmis_MURA.zip ...
Çıkartma işlemi tamamlandı.
Aktif İşlem Birimi: cuda


BAŞLA
    GEREKLİ MODÜLLERİ YÜKLE (Derin Öğrenme, Veri İşleme, Değerlendirme Metrikleri)
    
    Bulut Depolama Alanını (Google Drive) çalışma dizinine bağla
    
    ZİP_YOLU = "Drive'daki sıkıştırılmış verinin konumu"
    YEREL_YOL = "Sanal makinenin hızlı diski"
    
    EĞER YEREL_YOL mevcut DEĞİLSE:
        ZİP_YOLU'ndaki arşivi oku
        Arşiv içeriğini YEREL_YOL içerisine çıkart
        "Çıkartma işlemi tamamlandı" mesajını göster
    DEĞİLSE:
        "Veri seti hazır" mesajını göster
    
    EĞER Sistemde CUDA (GPU) desteği VARSA:
        ATAMA YAP: islem_birimi = "cuda"
    DEĞİLSE:
        ATAMA YAP: islem_birimi = "cpu"
        
    Kullanılacak islem_birimi'ni ekrana yazdır
BİTİR

In [ ]:
#Hücre 2: İzole Hiperparametre Konfigürasyonu

# ==============================================================================
# HİPERPARAMETRE VE DENEY YÖNETİM MERKEZİ (CONFIG SÖZLÜĞÜ)
# ==============================================================================
# Tüm model, eğitim ve veri artırımı ayarlarının merkezi olarak yönetildiği sözlük.
# Bu yapı, koda gömülü (hardcoded) değerleri engelleyerek deneyler arası adil
# kıyaslama yapılmasını ve hiperparametrelerin otomatik loglanmasını sağlar.

CONFIG = {
    # Deneyin kayıt ve takip ismi. Çıktı klasörleri ve dosya isimleri buna göre oluşturulur.
    "experiment_name": "Exp 2.1: MURA and Traditional CNN(resnet50)",

    # Kullanılacak temel konvolüsyonel sinir ağı (CNN) mimarisi. Jenerik model
    # oluşturucu fonksiyon (Hücre 4) bu değeri okuyarak ilgili ağırlıkları çeker.
    "model_architecture": "resnet50",

    # Görüntülerin modele girmeden önceki standart boyutu. ImageNet ön-eğitimli
    # modeller (ResNet, DenseNet vb.) optimum başarım için genellikle 224x224 formatını bekler.
    "target_image_size": (224, 224),

    # Her bir iterasyonda (forward/backward pass) işlenecek görüntü sayısı.
    # 32 değeri, GPU VRAM kapasitesi ile gradyan güncellemelerinin stabilitesi arasında ideal bir dengedir.
    "batch_size": 32,

    # Temel Öğrenme Oranı (Base Learning Rate). Transfer öğrenme (Fine-Tuning)
    # yapıldığı için önceden öğrenilmiş ağırlıkları bozmamak adına küçük bir değer seçilmiştir.
    "learning_rate": 5e-5,

    # Eğitimin maksimum süreceği toplam epok (veri setinin tam tur dönülmesi) sayısı.
    "num_epochs": 50,

    # L2 Regülarizasyon (Weight Decay) katsayısı. Modelin aşırı öğrenmesini (overfitting)
    # engellemek ve ağırlıkların aşırı büyümesini cezalandırmak için 5e-3 gibi güçlü bir değer seçilmiştir.
    "weight_decay": 5e-3,

    # Erken Durdurma (Early Stopping) toleransı. Doğrulama kaybı (Validation Loss)
    # belirtilen epok sayısı (12) boyunca iyileşmezse eğitim otomatik olarak sonlandırılır.
    "early_stopping_patience": 12,

    # MixUp Veri Artırımı stratejisinin durumu. Tıbbi görüntülerde (özellikle röntgenlerde)
    # kırık hatları lineer interpolasyonla bozulduğunda modelin kafası karışabildiği için kapalı (False) tutulmuştur.
    "use_mixup": False,

    # MixUp aktif olduğunda kullanılacak Beta dağılımı parametresi (Şu an pasif).
    "mixup_alpha": 0.2,

    # DataLoader'ın veriyi diskten RAM'e çekerken kullanacağı paralel alt işlem (subprocess) sayısı.
    "num_workers": 4,

    # ==========================================================================
    # DİNAMİK VERİ ARTIRIMI (ON-THE-FLY DATA AUGMENTATION) PARAMETRELERİ
    # ==========================================================================
    # JSON dosyasına yansıyacak ve Transforms (Hücre 3) bloğunda kullanılacak ayarlar.
    # Tıbbi verinin anatomik gerçekliğini bozmayacak sınırlar içerisinde belirlenmiştir.
    "augmentations": {
        "random_rotation_degrees": 15,    # Maksimum dönüş açısı (Tıbbi veride 10-15 derece idealdir)
        "horizontal_flip_prob": 0.5,      # %50 ihtimalle yatay çevirme (Anatomik simetriyi korur)
        "color_jitter_brightness": 0.2,   # Parlaklık değişim toleransı (Röntgen kontrast farklılıklarını simüle eder)
        "color_jitter_contrast": 0.2      # Kontrast değişim toleransı
    }
}


    TANIMLA YAPISAL_SÖZLÜK (CONFIG):
        Ekle "Deney Adı" -> "Exp 2.1: MURA and Traditional CNN(resnet50)"
        Ekle "Mimari" -> "resnet50"
        Ekle "Görüntü Çözünürlüğü" -> (224, 224)
        Ekle "Yığın Boyutu (Batch)" -> 32
        Ekle "Öğrenme Oranı (LR)" -> 0.00005
        Ekle "Maksimum Epok" -> 50
        Ekle "L2 Regülarizasyon (Weight Decay)" -> 0.005
        Ekle "Erken Durdurma Toleransı" -> 12
        Ekle "MixUp Kullanımı" -> YANLIŞ
        Ekle "MixUp Beta Değeri" -> 0.2
        Ekle "Paralel İşlemci Sayısı" -> 4
        
        TANIMLA ALT_SÖZLÜK "Veri Artırımı":
            Ekle "Döndürme Açısı" -> 15
            Ekle "Yatay Çevirme Olasılığı" -> %50
            Ekle "Parlaklık Sapması" -> 0.2
            Ekle "Kontrast Sapması" -> 0.2
            
    KAYDET YAPISAL_SÖZLÜK


In [ ]:
#Hücre 3: Jenerik Veri Yükleyici ve Dinamik Artırma

import os
import torch
import numpy as np
from PIL import Image
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as transforms

# ==============================================================================
# ÖZELLEŞTİRİLMİŞ TIBBİ GÖRÜNTÜ VERİ KÜMESİ SINIFI
# ==============================================================================
# PyTorch'un standart Dataset sınıfından türetilen bu yapı, MURA veri setinin
# karmaşık klasör hiyerarşisini dinamik olarak tarayarak görüntüleri ve etiketleri eşleştirir.
class JenerikMedikalDataset(Dataset):
    def __init__(self, root_dir, transform=None):
        self.root_dir = root_dir
        self.transform = transform
        self.image_paths = []
        self.labels = []

        # MURA veri setinin doğası gereği etiketler klasör/dosya isimlerinde gizlidir.
        # 'positive' = 1 (Kırık/Anormal), 'negative' = 0 (Sağlıklı/Normal)
        for root, dirs, files in os.walk(root_dir):
            for img_name in files:
                if img_name.lower().endswith(('.png', '.jpg', '.jpeg')):
                    tam_yol = os.path.join(root, img_name)
                    # Sınıflandırma problemi için etiket çıkarma (Label Extraction)
                    if 'positive' in tam_yol.lower():
                        self.image_paths.append(tam_yol)
                        self.labels.append(1)
                    elif 'negative' in tam_yol.lower():
                        self.image_paths.append(tam_yol)
                        self.labels.append(0)

    # Veri setindeki toplam örnek sayısını döndürür (Epoch iterasyonları için gereklidir)
    def __len__(self):
        return len(self.image_paths)

    # Verilen indeksteki görüntüyü diskten anlık olarak (on-the-fly) çeker.
    # Bu yöntem, tüm veri setini RAM'e yükleyip sistemi çökertmekten kurtarır.
    def __getitem__(self, idx):
        img_path = self.image_paths[idx]
        image = Image.open(img_path).convert('RGB')
        label = self.labels[idx]

        # Eğer augmentasyon veya normalizasyon tanımlanmışsa görüntüyü deforme eder
        if self.transform:
            image = self.transform(image)
        return image, label

# =====================================================================
# DİNAMİK VERİ ARTIRMA (ON-THE-FLY AUGMENTATION)
# =====================================================================
# Modelin ezberlemesini (overfitting) önlemek için eğitim verilerine her iterasyonda
# rastgele dönüşümler uygulanır. Değerler izole CONFIG sözlüğünden çekilir.
train_transforms = transforms.Compose([
    # Modellerin (ResNet vb.) beklediği standart girdi matris boyutuna ayarlama
    transforms.Resize(CONFIG["target_image_size"]),

    # 1. Rotasyon: Tıbbi çekim hatalarını ve açı farklarını simüle eder
    transforms.RandomRotation(CONFIG["augmentations"]["random_rotation_degrees"]),

    # 2. Çevirme: Anatomik simetriyi kullanarak veri havuzunu genişletir
    transforms.RandomHorizontalFlip(p=CONFIG["augmentations"]["horizontal_flip_prob"]),

    # 3. Parlaklık/Kontrast: Farklı röntgen cihazlarının X-ışını doz farklarını simüle eder
    transforms.ColorJitter(
        brightness=CONFIG["augmentations"]["color_jitter_brightness"],
        contrast=CONFIG["augmentations"]["color_jitter_contrast"]
    ),

    # Görüntüyü PyTorch tensörüne çevirir ve piksel değerlerini [0, 1] aralığına çeker
    transforms.ToTensor(),

    # ImageNet Ön-Eğitimli (Pre-trained) modellerin beklediği standart normalizasyon değerleri.
    # Modelin çok daha hızlı ve stabil yakınsamasını (convergence) sağlar.
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

# Doğrulama/Test setleri modelin asıl performansını ölçeceği için deformasyona uğratılmaz (Augmentation uygulanmaz)
# Sadece tensör dönüşümü ve ImageNet normalizasyonu yapılır.
val_transforms = transforms.Compose([
    transforms.Resize(CONFIG["target_image_size"]),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

# Klasör yolları (MURA v1.1 yapısına göre)
TRAIN_DIR = os.path.join(YEREL_VERI_KLASORU, 'MURA-v1.1', 'train')
VAL_DIR = os.path.join(YEREL_VERI_KLASORU, 'MURA-v1.1', 'valid')

# Veri seti nesnelerinin oluşturulması
train_dataset = JenerikMedikalDataset(root_dir=TRAIN_DIR, transform=train_transforms)
val_dataset = JenerikMedikalDataset(root_dir=VAL_DIR, transform=val_transforms)

# DataLoader: Verileri diskten GPU'ya belirlenen yığınlar (batch) halinde taşıyan köprü yapısı.
# pin_memory=True: CPU RAM'inden GPU VRAM'ine veri transferini hızlandırır.
train_loader = DataLoader(train_dataset, batch_size=CONFIG["batch_size"], shuffle=True, num_workers=CONFIG["num_workers"], pin_memory=True)
val_loader = DataLoader(val_dataset, batch_size=CONFIG["batch_size"], shuffle=False, num_workers=CONFIG["num_workers"], pin_memory=True)

print(f"Eğitim Verisi: {len(train_dataset)} | Doğrulama Verisi: {len(val_dataset)}")

# ==============================================================================
# MIXUP REGÜLARİZASYONU (İSTEĞE BAĞLI / ABLASYON)
# ==============================================================================
# İki farklı görüntüyü ve etiketini Beta dağılımı üzerinden şeffaf bir şekilde üst üste bindirerek
# modelin kesin karar sınırlarını yumuşatmayı hedefleyen ileri düzey veri artırımı tekniği.
def mixup_data(x, y, alpha=CONFIG["mixup_alpha"]):
    if alpha > 0:
        lam = np.random.beta(alpha, alpha)
    else:
        lam = 1
    batch_size = x.size()[0]
    index = torch.randperm(batch_size).to(x.device)

    # İki görüntünün piksellerini lambda (lam) oranında karıştırır
    mixed_x = lam * x + (1 - lam) * x[index, :]
    y_a, y_b = y, y[index]
    return mixed_x, y_a, y_b, lam

# Mixup işlemi uygulandığında standart kayıp fonksiyonunun da bu karışıma göre güncellenmesi gerekir.
def mixup_criterion(criterion, pred, y_a, y_b, lam):
    return lam * criterion(pred, y_a) + (1 - lam) * criterion(pred, y_b)

Eğitim Verisi: 36808 | Doğrulama Verisi: 3197


    BAŞLA
    TANIMLA SINIF "JenerikMedikalDataset":
        İLKLE (Klasör_Yolu, Dönüşümler):
            Ekle "Dosya Taraması" -> Ana klasördeki tüm alt klasörleri ve dosyaları tara
            EĞER dosya ismi "positive" içeriyorsa:
                Ekle "Kayıt" -> Listeye ekle (Dosya_Yolu, Etiket = 1)
            EĞER dosya ismi "negative" içeriyorsa:
                Ekle "Kayıt" -> Listeye ekle (Dosya_Yolu, Etiket = 0)
        
        ÖĞE_GETİR (İndeks):
            Ekle "Okuma" -> İndeksteki resmi diskten oku ve RGB formatına çevir
            EĞER Dönüşümler (Transform) TANIMLIYSA:
                Ekle "Deformasyon" -> Görüntüyü ilgili dönüşümlerden geçir
            DÖNDÜR Görüntü, Etiket

    TANIMLA DİZİ "Eğitim_Dönüşümleri" (train_transforms):
        Ekle "Boyutlandırma" -> CONFIG'den hedeflenen boyutu çek (224x224)
        Ekle "Rastgele Döndürme" -> CONFIG'den açı değerini çek
        Ekle "Yatay Çevirme" -> CONFIG'den ihtimal değerini çek
        Ekle "Renk Sapması" -> CONFIG'den parlaklık ve kontrast toleransını çek
        Ekle "Tensör Dönüşümü" -> Pikselleri [0, 1] aralığına sıkıştır
        Ekle "Normalizasyon" -> ImageNet ortalamalarıyla (Mean, Std) standartlaştır

    TANIMLA DİZİ "Doğrulama_Dönüşümleri" (val_transforms):
        Ekle "Boyutlandırma" -> CONFIG'den hedeflenen boyutu çek (224x224)
        Ekle "Tensör Dönüşümü" -> Pikselleri [0, 1] aralığına sıkıştır
        Ekle "Normalizasyon" -> ImageNet ortalamalarıyla standartlaştır (Deformasyon YOK)

    TANIMLA NESNE "train_dataset" -> JenerikMedikalDataset(Eğitim_Klasörü, Eğitim_Dönüşümleri)
    TANIMLA NESNE "val_dataset" -> JenerikMedikalDataset(Doğrulama_Klasörü, Doğrulama_Dönüşümleri)

    TANIMLA NESNE "train_loader" -> DataLoader(train_dataset, batch_size=CONFIG, Karıştır=DOĞRU)
    TANIMLA NESNE "val_loader" -> DataLoader(val_dataset, batch_size=CONFIG, Karıştır=YANLIŞ)

    TANIMLA FONKSİYON "mixup_data" (Görüntü, Etiket, Alpha):
        Ekle "Karışım Oranı" -> Beta dağılımından bir Lambda (lam) değeri üret
        Ekle "Harmanlama" -> Orijinal ve rastgele seçilmiş görüntüleri Lambda oranında üst üste bindir
        DÖNDÜR Harmanlanmış Görüntü, Etiket A, Etiket B, Lambda

    TANIMLA FONKSİYON "mixup_criterion" (Kayıp_Fonksiyonu, Tahmin, Etiket_A, Etiket_B, Lambda):
        DÖNDÜR Lambda ile ağırlıklandırılmış çapraz entropi (Cross-Entropy) kaybı
    BİTİR

In [ ]:
#Hücre 4: Kapsamlı Jenerik Model Oluşturucu

import torch
import torch.nn as nn
import torch.optim as optim
import torchvision.models as models

# ==============================================================================
# JENERİK MODEL OLUŞTURUCU FONKSİYON
# ==============================================================================
# Farklı CNN ve Transformer mimarilerini tek bir arayüzden çağırmak, son karar
# katmanlarını (classification head) ikili sınıflandırmaya uyarlamak ve
# aşırı öğrenmeyi önleyici (dropout) mekanizmaları eklemek için tasarlanmıştır.
def jenerik_model_olustur(model_adi, num_classes=2, dropout_rate=0.5):
    print(f"[{model_adi}] mimarisi ImageNet ağırlıklarıyla yükleniyor... (Dropout: {dropout_rate})")

    # --- RESNET AİLESİ ---
    if model_adi == "resnet50":
        model = models.resnet50(weights=models.ResNet50_Weights.IMAGENET1K_V2)
        model.fc = nn.Sequential(
            nn.Dropout(p=dropout_rate),
            nn.Linear(model.fc.in_features, num_classes)
        )

    elif model_adi == "resnet101":
        model = models.resnet101(weights=models.ResNet101_Weights.IMAGENET1K_V2)
        model.fc = nn.Sequential(
            nn.Dropout(p=dropout_rate),
            nn.Linear(model.fc.in_features, num_classes)
        )

    # --- DENSENET AİLESİ ---
    elif model_adi == "densenet121":
        model = models.densenet121(weights=models.DenseNet121_Weights.IMAGENET1K_V1)
        model.classifier = nn.Sequential(
            nn.Dropout(p=dropout_rate),
            nn.Linear(model.classifier.in_features, num_classes)
        )

    elif model_adi == "densenet201":
        model = models.densenet201(weights=models.DenseNet201_Weights.IMAGENET1K_V1)
        model.classifier = nn.Sequential(
            nn.Dropout(p=dropout_rate),
            nn.Linear(model.classifier.in_features, num_classes)
        )

    # --- EFFICIENTNET AİLESİ ---
    elif model_adi == "efficientnet_b4":
        model = models.efficientnet_b4(weights=models.EfficientNet_B4_Weights.IMAGENET1K_V1)
        model.classifier[1] = nn.Sequential(
            nn.Dropout(p=dropout_rate),
            nn.Linear(model.classifier[1].in_features, num_classes)
        )

    elif model_adi == "efficientnet_v2_s":
        model = models.efficientnet_v2_s(weights=models.EfficientNet_V2_S_Weights.IMAGENET1K_V1)
        model.classifier[1] = nn.Sequential(
            nn.Dropout(p=dropout_rate),
            nn.Linear(model.classifier[1].in_features, num_classes)
        )

    # --- CONVNEXT AİLESİ ---
    elif model_adi == "convnext_base":
        model = models.convnext_base(weights=models.ConvNeXt_Base_Weights.IMAGENET1K_V1)
        model.classifier[2] = nn.Sequential(
            nn.Dropout(p=dropout_rate),
            nn.Linear(model.classifier[2].in_features, num_classes)
        )

    # --- MOBILENET ---
    elif model_adi == "mobilenet_v3_large":
        model = models.mobilenet_v3_large(weights=models.MobileNet_V3_Large_Weights.IMAGENET1K_V2)
        model.classifier[3] = nn.Sequential(
            nn.Dropout(p=dropout_rate),
            nn.Linear(model.classifier[3].in_features, num_classes)
        )

    # --- VGG AİLESİ ---
    elif model_adi == "vgg19_bn":
        model = models.vgg19_bn(weights=models.VGG19_BN_Weights.IMAGENET1K_V1)
        model.classifier[6] = nn.Sequential(
            nn.Dropout(p=dropout_rate),
            nn.Linear(model.classifier[6].in_features, num_classes)
        )

    # ==========================================
    # --- VISION TRANSFORMERS (ViT & SWIN) ---
    # ==========================================
    elif model_adi == "vit_b_16":
        model = models.vit_b_16(weights=models.ViT_B_16_Weights.IMAGENET1K_V1)
        model.heads.head = nn.Sequential(
            nn.Dropout(p=dropout_rate),
            nn.Linear(model.heads.head.in_features, num_classes)
        )

    elif model_adi == "swin_b":
        model = models.swin_b(weights=models.Swin_B_Weights.IMAGENET1K_V1)
        model.head = nn.Sequential(
            nn.Dropout(p=dropout_rate),
            nn.Linear(model.head.in_features, num_classes)
        )

    else:
        raise ValueError(f"HATA: '{model_adi}' tanımlı bir model değil. Lütfen CONFIG sözlüğünü kontrol edin.")

# ==========================================================
    # JENERİK KATMAN DONDURMA (TRANSFER LEARNING / FINE-TUNING STRATEJİSİ)
    # ==========================================================
    # Modelin ImageNet'te öğrendiği temel özellikleri (kenar, doku vb.) kaybetmesini
    # (catastrophic forgetting) önlemek ve MURA gibi sınırlı bir veri setinde ezberlemeyi
    # (overfitting) durdurmak için modelin ilk katmanları dondurulur (requires_grad = False).
    dondurulan_katman_sayisi = 0
    acik_katman_sayisi = 0

    for name, param in model.named_parameters():
        # Sadece üst düzey özellik çıkaran gövde katmanları (layer3, layer4) ve
        # yeni eklediğimiz karar katmanları (fc, classifier, head) eğitime açık bırakılır.
        if any(keyword in name for keyword in ["layer3", "layer4", "fc", "classifier", "head"]):
            param.requires_grad = True
            acik_katman_sayisi += 1
        else:
            param.requires_grad = False
            dondurulan_katman_sayisi += 1

    print(f"Transfer Learning Stratejisi: {dondurulan_katman_sayisi} katman donduruldu, {acik_katman_sayisi} katman eğitiliyor.")

    return model.to(device)

# Modeli başlatma
model = jenerik_model_olustur(CONFIG["model_architecture"], dropout_rate=0.5)

# ==========================================================
# FARKILAŞTIRILMIŞ ÖĞRENME ORANI (DISCRIMINATIVE FINE-TUNING)
# ==========================================================
# Ön-eğitimli gövde (backbone) ağırlıkları ile sıfırdan oluşturulan başlık (head)
# ağırlıklarının aynı hızda güncellenmesi, gövdedeki değerli ön bilgiyi bozacaktır.
head_params = []
backbone_params = []

for name, param in model.named_parameters():
    if not param.requires_grad:
        continue # Dondurulmuş katmanları atla

    # Eğer katman yeni eklediğimiz sınıflandırıcı başlık ise
    if any(keyword in name for keyword in ["fc", "classifier", "head"]):
        head_params.append(param)
    # Eğer katman ön-eğitimli gövde (layer3, layer4 vb.) ise
    else:
        backbone_params.append(param)

# Gövdeye CONFIG'deki LR'nin 10'da 1'i verilir (Ön bilgiyi korumak için çok hassas adımlar)
# Başlığa CONFIG'deki normal LR verilir (Hızlı öğrenmesi için normal adımlar)
optimizer = optim.AdamW([
    {'params': backbone_params, 'lr': CONFIG["learning_rate"] / 10},
    {'params': head_params, 'lr': CONFIG["learning_rate"]}
], weight_decay=CONFIG["weight_decay"])

# ==========================================================
# AĞIRLIKLI KAYIP FONKSİYONU (CLASS IMBALANCE HANDLING)
# ==========================================================
# MURA veri setindeki pozitif (kırık) ve negatif (sağlam) sınıflar arasındaki
# dengesizliği aşmak için, azınlık/kritik olan "kırık" sınıfına 1.5 kat ceza ağırlığı atanır.
class_weights = torch.tensor([1.0, 1.5]).to(device)
criterion = nn.CrossEntropyLoss(weight=class_weights)

print(f"Model başarıyla GPU'ya ({device}) taşındı ve eğitime hazır.")

[resnet50] mimarisi ImageNet ağırlıklarıyla yükleniyor... (Dropout: 0.5)
Downloading: "https://download.pytorch.org/models/resnet50-11ad3fa6.pth" to /root/.cache/torch/hub/checkpoints/resnet50-11ad3fa6.pth


100%|██████████| 97.8M/97.8M [00:00<00:00, 158MB/s]


Transfer Learning Stratejisi: 72 katman donduruldu, 89 katman eğitiliyor.
Model başarıyla GPU'ya (cuda) taşındı ve eğitime hazır.


    BAŞLA
    TANIMLA FONKSİYON "jenerik_model_olustur" (Model_Adı, Sınıf_Sayısı, Dropout_Oranı):
        Ekle "Model Yükleme" -> Belirtilen mimariyi ImageNet ağırlıklarıyla yükle
        Ekle "Başlık Uyarlaması" -> Mimarinin son karar katmanını tespit et (fc, classifier veya head)
        Ekle "Düzenlileştirme" -> Son katmanı Dropout (%50) ve Sınıf_Sayısı (2) çıktılı Doğrusal (Linear) katman ile değiştir
        
        Ekle "Katman Dondurma Döngüsü" -> Modelin tüm katmanları üzerinde sırayla gezin:
            EĞER katman ismi "layer3", "layer4", "fc", "classifier" veya "head" içeriyorsa:
                Ekle "Eğitime Açık" -> Katmanın gradyan güncellemesini aktifleştir (requires_grad = True)
            DEĞİLSE:
                Ekle "Eğitime Kapalı" -> Katmanı dondur (requires_grad = False)
                
        DÖNDÜR İşlemleri tamamlanan model
        
    TANIMLA NESNE "model" -> jenerik_model_olustur(CONFIG içerisindeki mimari ismi)
    
    TANIMLA LİSTE "Gövde_Parametreleri"
    TANIMLA LİSTE "Başlık_Parametreleri"
    
    Ekle "Parametre Ayrıştırma Döngüsü" -> Modelin eğitime açık katmanları üzerinde gezin:
        EĞER katman ismi yeni başlık ise (fc, classifier, head):
            Ekle "Başlık Grubu" -> Katmanı Başlık_Parametreleri listesine ekle
        DEĞİLSE:
            Ekle "Gövde Grubu" -> Katmanı Gövde_Parametreleri listesine ekle
            
    TANIMLA NESNE "optimizer" (AdamW Algoritması):
        Ekle "Gövde Hızı" -> Gövde_Parametreleri için CONFIG'deki öğrenme oranını 10'a bölerek ata
        Ekle "Başlık Hızı" -> Başlık_Parametreleri için CONFIG'deki normal öğrenme oranını ata
        Ekle "L2 Cezası" -> CONFIG'deki weight_decay değerini optimizasyona dahil et
        
    TANIMLA NESNE "criterion" (Ağırlıklı Çapraz Entropi Kaybı):
        Ekle "Sınıf Ağırlıkları" -> Negatif sınıfa 1.0, Pozitif sınıfa 1.5 ceza katsayısı ver
        Ekle "Kayıp Fonksiyonu" -> Bu ağırlıkları hata hesaplama fonksiyonuna entegre et
    BİTİR

In [ ]:
#Hücre 5: Kapsamlı Başarım Metrikleri Hesaplayıcısı

# ==============================================================================
# KAPSAMLI BAŞARIM METRİKLERİ HESAPLAYICISI
# ==============================================================================
# Modelin ürettiği tahminleri (predictions) ve olasılık skorlarını (probabilities)
# gerçek etiketlerle (ground truth) karşılaştırarak, tıbbi tanı sistemleri için
# literatürde kabul gören tüm değerlendirme ölçütlerini tek seferde hesaplar.

def kapsamli_metrikleri_hesapla(y_true, y_pred, y_scores):
    # Karmaşıklık Matrisi (Confusion Matrix) hesaplaması
    # tn (True Negative), fp (False Positive), fn (False Negative), tp (True Positive)
    cm = confusion_matrix(y_true, y_pred)
    tn, fp, fn, tp = cm.ravel() if cm.size == 4 else (0,0,0,0)

    # Scikit-learn fonksiyonları kullanılarak metriklerin sözlük yapısında toplanması
    metrikler = {
        # Genel doğruluk oranı (Sınıf dengesizliği olan durumlarda tek başına yanıltıcı olabilir)
        "Accuracy": accuracy_score(y_true, y_pred),

        # Kesinlik: Modelin "Kırık (1)" dediklerinin ne kadarının gerçekten kırık olduğu
        "Precision": precision_score(y_true, y_pred, zero_division=0),

        # Duyarlılık (Recall/Sensitivity): Gerçekte "Kırık" olanların ne kadarının model tarafından bulunabildiği
        "Recall_Sensitivity": recall_score(y_true, y_pred, zero_division=0),

        # Özgüllük (Specificity): Gerçekte "Sağlıklı" olanların ne kadarının doğru tespit edildiği
        "Specificity": tn / (tn + fp) if (tn + fp) > 0 else 0,

        # F1-Skoru: Kesinlik ve Duyarlılık metriklerinin harmonik ortalaması
        "F1_Measure": f1_score(y_true, y_pred, zero_division=0),

        # Cohen's Kappa: Şans eseri oluşan doğru tahminleri cezalandıran, Stanford'ın MURA
        # yarışmasında ana sıralama (leaderboard) ölçütü olarak kullandığı en kritik metrik.
        "Cohen_Kappa": cohen_kappa_score(y_true, y_pred),

        # ROC-AUC: Alıcı İşletim Karakteristiği Eğrisi Altında Kalan Alan (Modelin sınıfları ayırma gücü)
        "ROC_AUC": roc_auc_score(y_true, y_scores),

        # PR-AUC (uAP): Kesinlik-Duyarlılık Eğrisi Altında Kalan Alan. Dengesiz veri setlerinde
        # ROC-AUC'ye kıyasla model başarısını çok daha objektif yansıtır.
        "PR_AUC_uAP": average_precision_score(y_true, y_scores), # Uninterpolated Average Precision

        # Hata Metrikleri: Tahmin edilen olasılıklar ile gerçek etiketler (0 veya 1) arasındaki mutlak ve karesel sapmalar
        "MAE": mean_absolute_error(y_true, y_scores),
        "RMSE": np.sqrt(mean_squared_error(y_true, y_scores))
    }
    return metrikler

    BAŞLA
    TANIMLA FONKSİYON "kapsamli_metrikleri_hesapla" (Gerçek_Etiketler, Tahmin_Sınıfları, Tahmin_Olasılıkları):
        Ekle "Karmaşıklık Matrisi (CM)" -> Gerçek ve tahmin edilen sınıfları karşılaştırarak matrisi hesapla
        Ekle "Matris Ayrıştırma" -> CM üzerinden TN, FP, FN ve TP (Doğru/Yanlış, Pozitif/Negatif) değerlerini çıkar
        
        TANIMLA SÖZLÜK "metrikler":
            Ekle "Accuracy" -> Genel doğruluk oranını hesapla
            Ekle "Precision" -> Kesinlik skorunu hesapla (Sıfıra bölünme hatasını engelle)
            Ekle "Recall_Sensitivity" -> Duyarlılık skorunu hesapla
            Ekle "Specificity" -> Formül (TN / (TN + FP)) ile özgüllük skorunu hesapla
            Ekle "F1_Measure" -> Kesinlik ve Duyarlılığın harmonik ortalamasını (F1) hesapla
            Ekle "Cohen_Kappa" -> Şans eseri başarıyı denklemden çıkartan Kappa skorunu hesapla
            Ekle "ROC_AUC" -> Sınıflandırma yeteneğini gösteren ROC eğrisi alanını hesapla
            Ekle "PR_AUC_uAP" -> Dengesiz sınıflar için Kesinlik-Duyarlılık eğrisi alanını hesapla
            Ekle "MAE" -> Tahmin ile gerçeklik arasındaki Ortalama Mutlak Hatayı hesapla
            Ekle "RMSE" -> Hataların karesel ortalamasının karekökünü hesapla
            
        DÖNDÜR Hesaplanan metrikler sözlüğünü
    BİTİR

In [ ]:
import time
import os
import pandas as pd
import torch
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import json
from tqdm import tqdm
from sklearn.metrics import confusion_matrix, roc_curve, auc, precision_recall_curve

# ==============================================================================
# EĞİTİM DÖNGÜSÜ HAZIRLIKLARI VE İZLEME DEĞİŞKENLERİ
# ==============================================================================
best_val_loss = float('inf') # En iyi modelin kaydedilmesi için başlangıç eşiği
patience_counter = 0         # Erken durdurma (Early Stopping) için sayaç
egitim_gecmisi = []          # Her epok sonunda elde edilen metriklerin toplanacağı liste

# --- SCHEDULER (ÖĞRENME ORANI PLANLAYICI) ---
# Modelin doğrulama kaybı (val loss) 3 epok boyunca iyileşmezse, öğrenme oranını
# yarıya (%50) düşürerek (factor=0.5) daha ince ayar yapmasını sağlar.
# "min" modu, hedefin kaybı küçültmek olduğunu belirtir.
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=3)

print(f"[{CONFIG['experiment_name']}] Eğitim Başlıyor...")
toplam_baslangic_zamani = time.time()

# ==============================================================================
# ANA EPOK DÖNGÜSÜ
# ==============================================================================
for epoch in range(CONFIG["num_epochs"]):
    epoch_baslangic_zamani = time.time()

    # --- 1. EĞİTİM (TRAINING) FAZI ---
    model.train() # Modeli eğitim moduna alır (Dropout ve BatchNorm katmanları aktifleşir)
    train_loss = 0.0

    loop = tqdm(train_loader, desc=f"Epoch {epoch+1}/{CONFIG['num_epochs']} - Eğitim")
    for inputs, labels in loop:
        inputs, labels = inputs.to(device), labels.to(device)
        optimizer.zero_grad() # Önceki iterasyondan kalan gradyanları sıfırla

        # İleri Yönlü Yayılım (Forward Pass) ve MixUp Veri Artırımı Kontrolü
        if CONFIG.get("use_mixup", False):
            inputs, targets_a, targets_b, lam = mixup_data(inputs, labels)
            outputs = model(inputs)
            loss = mixup_criterion(criterion, outputs, targets_a, targets_b, lam)
        else:
            outputs = model(inputs)
            loss = criterion(outputs, labels)

        # Geri Yayılım (Backward Pass) ve Ağırlık Güncellemesi
        loss.backward()
        optimizer.step()
        train_loss += loss.item() * inputs.size(0)

        # İzleme çubuğuna (tqdm) farklılaştırılmış öğrenme oranlarının anlık yansıtılması
        lr_backbone = optimizer.param_groups[0]['lr']
        lr_head = optimizer.param_groups[-1]['lr']
        loop.set_postfix(loss=loss.item(), lr_govde=lr_backbone, lr_baslik=lr_head)

    # Epok sonu ortalama eğitim kaybının hesaplanması
    epoch_train_loss = train_loss / len(train_loader.dataset)

    # --- 2. DOĞRULAMA (VALIDATION) FAZI ---
    model.eval() # Modeli doğrulama moduna alır (Ağırlık güncellemeleri durur)
    val_loss = 0.0
    y_true, y_pred, y_scores = [], [], []
    toplam_cikarim_suresi = 0.0
    ornek_sayisi = 0

    with torch.no_grad(): # Hafıza tasarrufu için gradyan hesaplamasını tamamen kapat
        for inputs, labels in tqdm(val_loader, desc="Doğrulama"):
            inputs, labels = inputs.to(device), labels.to(device)

            # Çıkarım süresi (Inference Time) ölçümü (Klinik hız analizi için)
            start_infer = time.time()
            outputs = model(inputs)
            end_infer = time.time()

            toplam_cikarim_suresi += (end_infer - start_infer)
            ornek_sayisi += inputs.size(0)

            loss = criterion(outputs, labels)
            val_loss += loss.item() * inputs.size(0)

            # Model çıktılarının olasılıklara (softmax) ve kesin sınıflara (argmax) dönüştürülmesi
            probs = torch.softmax(outputs, dim=1)
            _, preds = torch.max(outputs, 1)

            # İstatistiksel hesaplamalar için değerlerin CPU'ya aktarılıp listelerde toplanması
            y_true.extend(labels.cpu().numpy())
            y_pred.extend(preds.cpu().numpy())
            y_scores.extend(probs[:, 1].cpu().numpy())

    # Epok sonu ortalama doğrulama kaybı ve milisaniye cinsinden çıkarım hızı
    epoch_val_loss = val_loss / len(val_loader.dataset)
    ms_per_step = (toplam_cikarim_suresi / ornek_sayisi) * 1000

    # --- 3. DİNAMİK OPTİMİZASYON VE KAYIT İŞLEMLERİ ---

    # Her epoch sonunda doğrulama kaybını scheduler'a bildirerek LR'nin ayarlanması
    scheduler.step(epoch_val_loss)

    # Hücre 5'teki fonksiyon çağrılarak tüm literatür metriklerinin hesaplanması
    metrikler = kapsamli_metrikleri_hesapla(y_true, y_pred, y_scores)

    epoch_bitis_zamani = time.time()
    epoch_suresi_sn = epoch_bitis_zamani - epoch_baslangic_zamani

    # Konsol Çıktıları
    # Not: current_lr değişkeni önceki kodlardan kalan bir kalıntı olabilir,
    # lr_head veya lr_backbone kullanımı daha doğru olurdu, ancak kod değiştirilmedi.
    current_lr = optimizer.param_groups[-1]['lr'] # Mevcut başlık öğrenme oranını okur
    print(f"\n--- Epoch {epoch+1} Sonuçları ---")
    print(f"Train Loss: {epoch_train_loss:.4f} | Val Loss: {epoch_val_loss:.4f} | Süre: {epoch_suresi_sn:.2f} sn | LR: {current_lr:.6f}")
    print(f"Accuracy: {metrikler['Accuracy']:.4f} | F1-Measure: {metrikler['F1_Measure']:.4f} | Kappa: {metrikler['Cohen_Kappa']:.4f}")
    print(f"PR-AUC (uAP): {metrikler['PR_AUC_uAP']:.4f} | ROC-AUC: {metrikler['ROC_AUC']:.4f}")
    print(f"Specificity: {metrikler['Specificity']:.4f} | Inference Time: {ms_per_step:.2f} ms/image")

    # Epok sonuçlarının genel listeye (Pandas DataFrame altyapısı) eklenmesi
    metrikler["Epoch"] = epoch + 1
    metrikler["Train_Loss"] = epoch_train_loss
    metrikler["Val_Loss"] = epoch_val_loss
    metrikler["Inference_Time_ms"] = ms_per_step
    metrikler["Epoch_Suresi_sn"] = epoch_suresi_sn
    metrikler["Learning_Rate"] = current_lr
    egitim_gecmisi.append(metrikler)

    # ==========================================================================
    # ERKEN DURDURMA (EARLY STOPPING) VE MODEL KAYDETME (CHECKPOINTING)
    # ==========================================================================
    # Aşırı öğrenmeyi (Overfitting) engellemek için, modelin ezberlemeden
    # genellenebilir örüntüler öğrendiği o altın "tepe noktası" diske kaydedilir.
    if epoch_val_loss < best_val_loss:
        best_val_loss = epoch_val_loss
        patience_counter = 0
        torch.save(model.state_dict(), f"/content/best_model_{CONFIG['model_architecture']}.pth")
    else:
        patience_counter += 1
        if patience_counter >= CONFIG["early_stopping_patience"]:
            print(f"\nErken Durdurma tetiklendi! {CONFIG['early_stopping_patience']} epoch boyunca Val Loss iyileşmedi.")
            break

toplam_bitis_zamani = time.time()
toplam_sure_dk = (toplam_bitis_zamani - toplam_baslangic_zamani) / 60
print(f"\nToplam Eğitim Süresi: {toplam_sure_dk:.2f} dakika.")

# ==============================================================================
# ÇIKTILARI, GRAFİKLERİ VE HİPERPARAMETRELERİ KAYDETME BÖLÜMÜ
# ==============================================================================
print("\nSonuçlar, grafikler ve hiperparametreler Google Drive'a kaydediliyor...")

# Drive üzerinde bu deneye özel benzersiz bir sonuç klasörü oluşturulur
grafik_klasoru = f"/content/drive/MyDrive/Doktora/Verisetleri_tik/deneyler/{CONFIG['experiment_name']}_Sonuclar"
os.makedirs(grafik_klasoru, exist_ok=True)

# 1. Epok epok tüm eğitim geçmişi sayısal formatta (CSV) kaydedilir
df_sonuclar = pd.DataFrame(egitim_gecmisi)
df_sonuclar.to_csv(os.path.join(grafik_klasoru, "Egitim_Metrikleri.csv"), index=False)

# 2. Deneyin birebir tekrar üretilebilmesi için CONFIG ayarları JSON olarak kaydedilir
kayit_verisi = CONFIG.copy()
kayit_verisi["Zaman_Bilgileri"] = {
    "Toplam_Egitim_Suresi_Dakika": round(toplam_sure_dk, 2),
    "Epoch_Sureleri_Saniye": [round(m["Epoch_Suresi_sn"], 2) for m in egitim_gecmisi]
}
kayit_verisi["Learning_Rate_Gecmisi"] = [m["Learning_Rate"] for m in egitim_gecmisi]

hiperparametre_dosyasi = os.path.join(grafik_klasoru, "Hiperparametreler.json")
with open(hiperparametre_dosyasi, "w", encoding="utf-8") as f:
    json.dump(kayit_verisi, f, indent=4, ensure_ascii=False)

# --- MATPLOTLIB İLE GÖRSELLEŞTİRME (KAYIP VE DOĞRULUK EĞRİLERİ) ---
# 1. Eğitim ve Doğrulama Kaybı (Loss) Eğrisi
plt.figure(figsize=(10, 5))
plt.plot(df_sonuclar['Epoch'], df_sonuclar['Train_Loss'], label='Training Loss', marker='o', markersize=4)
plt.plot(df_sonuclar['Epoch'], df_sonuclar['Val_Loss'], label='Validation Loss', marker='o', markersize=4)
plt.title('Training and Validation Loss')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.legend()
plt.grid(True, linestyle='--', alpha=0.7)
plt.tight_layout()
plt.savefig(os.path.join(grafik_klasoru, "1_Training_Validation_Loss_Curve.png"), dpi=300)
plt.close()

# 2. Accuracy Eğrisi
plt.figure(figsize=(10, 5))
plt.plot(df_sonuclar['Epoch'], df_sonuclar['Accuracy'], label='Accuracy Curve', color='green', marker='o', markersize=4)
plt.title('Accuracy')
plt.xlabel('Epoch')
plt.ylabel('Accuracy')
plt.legend()
plt.grid(True, linestyle='--', alpha=0.7)
plt.tight_layout()
plt.savefig(os.path.join(grafik_klasoru, "2_Accuracy_Curve.png"), dpi=300)
plt.close()

# ==============================================================================
# KRİTİK DÜZELTME: EN İYİ MODELİ GERİ YÜKLEME VE YENİDEN TAHMİN (BEST MODEL INFERENCE)
# ==============================================================================
# Eğer eğitim "Early Stopping" ile kesilirse, o an RAM'de bulunan model aslında
# ezberlemeye başlamış olan "kötü" modeldir. Nihai grafikleri (Confusion Matrix vb.)
# çizerken yanıltıcı sonuçlar almamak için, daha önce diske kaydedilen o "altın"
# ağırlıklar (best_model) geri yüklenir ve validasyon seti üzerinden tekrar test edilir.
print("\nGrafikler için 'En İyi Model (Best Model)' ağırlıkları geri yükleniyor...")
model.load_state_dict(torch.load(f"/content/best_model_{CONFIG['model_architecture']}.pth"))
model.eval()

y_true_best, y_pred_best, y_scores_best = [], [], []

# Sadece doğrulama seti üzerinden en iyi ağırlıklarla tekrar tahmin (inference) alıyoruz
with torch.no_grad():
    for inputs, labels in tqdm(val_loader, desc="Best Model Değerlendirmesi"):
        inputs, labels = inputs.to(device), labels.to(device)
        outputs = model(inputs)
        probs = torch.softmax(outputs, dim=1)
        _, preds = torch.max(outputs, 1)

        y_true_best.extend(labels.cpu().numpy())
        y_pred_best.extend(preds.cpu().numpy())
        y_scores_best.extend(probs[:, 1].cpu().numpy())

# --- NİHAİ BAŞARIM GRAFİKLERİ (BEST MODEL ÜZERİNDEN) ---
# 3. Karmaşıklık Matrisi (Heatmap)
cm_default = confusion_matrix(y_true_best, y_pred_best)
tn, fp, fn, tp = cm_default.ravel() if cm_default.size == 4 else (0, 0, 0, 0)
cm_custom = np.array([[tp, fp],
                      [fn, tn]])
plt.figure(figsize=(8, 6))
sns.heatmap(cm_custom, annot=True, fmt='d', cmap='Wistia', cbar=False,
            annot_kws={"size": 16, "weight": "bold"},
            xticklabels=['Actual Positive (1)', 'Actual Negative (0)'],
            yticklabels=['Predicted Positive (1)', 'Predicted Negative (0)'],
            linewidths=1, linecolor='black')
plt.title('Confusion Matrix', fontsize=16, fontweight='bold', pad=15)
plt.xlabel('Actual Values', fontsize=14, fontweight='bold', labelpad=10)
plt.ylabel('Predicted Values', fontsize=14, fontweight='bold', labelpad=10)
plt.yticks(rotation=90, va="center")
plt.tight_layout()
plt.savefig(os.path.join(grafik_klasoru, "3_Confusion_Matrix.png"), dpi=300)
plt.close()

# 4. ROC Eğrisi
fpr, tpr, _ = roc_curve(y_true_best, y_scores_best)
roc_auc = auc(fpr, tpr)
plt.figure(figsize=(7, 6))
plt.plot(fpr, tpr, color='darkorange', lw=2, label=f'ROC Curve (AUC = {roc_auc:.3f})')
plt.plot([0, 1], [0, 1], color='navy', lw=2, linestyle='--')
plt.xlim([0.0, 1.0])
plt.ylim([0.0, 1.05])
plt.xlabel('False Positive Rate (FPR)')
plt.ylabel('True Positive Rate (TPR)')
plt.title('Receiver Operating Characteristic (ROC)')
plt.legend(loc="lower right")
plt.grid(True, linestyle='--', alpha=0.5)
plt.tight_layout()
plt.savefig(os.path.join(grafik_klasoru, "4_ROC_Curve.png"), dpi=300)
plt.close()

# 5. Kesinlik-Duyarlılık (Precision-Recall) Eğrisi
precision, recall, _ = precision_recall_curve(y_true_best, y_scores_best)
plt.figure(figsize=(7, 6))
plt.plot(recall, precision, color='purple', lw=2)
plt.xlabel('Recall')
plt.ylabel('Precision')
plt.title('Precision-Recall Curve')
plt.grid(True, linestyle='--', alpha=0.5)
plt.tight_layout()
plt.savefig(os.path.join(grafik_klasoru, "5_PR_Curve.png"), dpi=300)
plt.close()

print(f"Tüm grafikler başarıyla '{grafik_klasoru}' klasörüne kaydedildi.")

[Exp 2.1: MURA and Traditional CNN(resnet50)] Eğitim Başlıyor...


Doğrulama: 100%|██████████| 100/100 [00:02<00:00, 45.70it/s]



--- Epoch 1 Sonuçları ---
Train Loss: 0.6317 | Val Loss: 0.5778 | Süre: 42.02 sn | LR: 0.000005
Accuracy: 0.7100 | F1-Measure: 0.6406 | Kappa: 0.4114
PR-AUC (uAP): 0.7971 | ROC-AUC: 0.7834
Specificity: 0.8662 | Inference Time: 0.40 ms/image


Doğrulama: 100%|██████████| 100/100 [00:02<00:00, 45.18it/s]



--- Epoch 2 Sonuçları ---
Train Loss: 0.5417 | Val Loss: 0.5276 | Süre: 39.69 sn | LR: 0.000005
Accuracy: 0.7382 | F1-Measure: 0.6811 | Kappa: 0.4692
PR-AUC (uAP): 0.8331 | ROC-AUC: 0.8232
Specificity: 0.8794 | Inference Time: 0.40 ms/image


...


--- Epoch 27 Sonuçları ---
Train Loss: 0.3841 | Val Loss: 0.4530 | Süre: 39.88 sn | LR: 0.000001
Accuracy: 0.8120 | F1-Measure: 0.7896 | Kappa: 0.6213
PR-AUC (uAP): 0.8881 | ROC-AUC: 0.8774
Specificity: 0.8806 | Inference Time: 0.36 ms/image


Doğrulama: 100%|██████████| 100/100 [00:02<00:00, 43.15it/s]



--- Epoch 28 Sonuçları ---
Train Loss: 0.3815 | Val Loss: 0.4617 | Süre: 40.09 sn | LR: 0.000001
Accuracy: 0.8129 | F1-Measure: 0.7916 | Kappa: 0.6233
PR-AUC (uAP): 0.8869 | ROC-AUC: 0.8755
Specificity: 0.8776 | Inference Time: 0.33 ms/image

Erken Durdurma tetiklendi! 12 epoch boyunca Val Loss iyileşmedi.

Toplam Eğitim Süresi: 18.86 dakika.

Sonuçlar, grafikler ve hiperparametreler Google Drive'a kaydediliyor...

Grafikler için 'En İyi Model (Best Model)' ağırlıkları geri yükleniyor...


Best Model Değerlendirmesi: 100%|██████████| 100/100 [00:02<00:00, 48.76it/s]


Tüm grafikler başarıyla '/content/drive/MyDrive/Doktora/Verisetleri_tik/deneyler/Exp 2.1: MURA and Traditional CNN(resnet50)_Sonuclar' klasörüne kaydedildi.


    BAŞLA
    TANIMLA Değişkenler: En_İyi_Kayıp = Sonsuzluk, Sabır_Sayacı = 0, Geçmiş_Listesi = []
    TANIMLA NESNE "scheduler" -> Doğrulama kaybı 3 epok düşmezse öğrenme hızını yarıya indir

    Ekle "Kronometre" -> Toplam eğitim süresini hesaplamak için sayacı başlat

    DÖNGÜ BAŞLAT "Ana Epok Döngüsü" (Maksimum Epok Sayısı Kadar):
    Ekle "Epok Kronometresi" -> Epok süresini başlat
    Ekle "Mod Değişimi" -> Modeli Eğitim (Train) moduna al
    
    DÖNGÜ BAŞLAT "Eğitim İterasyonları" (Eğitim Verisi Üzerinde):
        Ekle "Sıfırlama" -> Geçmiş iterasyonun gradyanlarını temizle (zero_grad)
        
        EĞER MixUp Aktifse:
            Ekle "Veri Artırımı" -> Görüntüleri ve etiketleri birbirine harmanla
            Ekle "İleri Yayılım" -> Modeli çalıştır ve MixUp uyumlu kaybı (loss) hesapla
        DEĞİLSE:
            Ekle "İleri Yayılım" -> Modeli çalıştır ve standart kaybı hesapla
            
        Ekle "Geri Yayılım" -> Hata payını ağ üzerinde geriye doğru dağıt (backward)
        Ekle "Optimizasyon" -> Öğrenme hızıyla ağırlıkları güncelle (step)
        Ekle "Kayıt" -> Gövde ve başlık öğrenme oranlarını ilerleme çubuğuna yansıt
        
    Ekle "Mod Değişimi" -> Modeli Doğrulama (Eval) moduna al (Güncellemeleri kapat)
    
    DÖNGÜ BAŞLAT "Doğrulama İterasyonları" (Doğrulama Verisi Üzerinde):
        Ekle "Çıkarım" -> Görüntüleri modele sok ve geçen klinik süreyi (Inference Time) ölç
        Ekle "Hata Hesaplama" -> Doğrulama kaybını (Val Loss) hesapla
        Ekle "Liste Kaydı" -> Gerçek etiketleri, tahminleri ve olasılıkları listelerde topla
        
    Ekle "Planlayıcı Güncellemesi" -> Doğrulama kaybını scheduler nesnesine bildir
    Ekle "Metrik Hesaplama" -> Hücre 5'teki fonksiyonu çağırarak literatür metriklerini elde et
    Ekle "Geçmiş Kaydı" -> Epok sonuçlarını Geçmiş_Listesi isimli tabloya ekle
    
    EĞER Bu_Epokun_Kayıp_Değeri < En_İyi_Kayıp:
        Ekle "Güncelleme" -> En_İyi_Kayıp değerini yenile ve Sabır_Sayacı'nı sıfırla
        Ekle "Model Kaydı" -> Modelin ağırlıklarını (best_model) yerel diske kaydet
    DEĞİLSE:
        Ekle "Sabır" -> Sabır_Sayacı'nı 1 artır
        EĞER Sabır_Sayacı >= Erken_Durdurma_Toleransı:
            Ekle "Durdurma" -> Erken durdurmayı tetikle ve ana döngüden çık
            
    Ekle "Sonlandırma" -> Toplam eğitim süresini hesapla ve ekrana yazdır

    Ekle "Dışa Aktarma" -> Geçmiş_Listesi'ni CSV, CONFIG sözlüğünü JSON formatında Drive'a kaydet
    Ekle "Görselleştirme 1" -> Eğitim/Doğrulama kaybı ve Doğruluk (Accuracy) geçmişini çizip kaydet

    Ekle "Kritik Düzeltme" -> Diske kaydedilmiş "best_model" ağırlıklarını modele geri yükle
    Ekle "Nihai Çıkarım" -> Doğrulama seti üzerinde o en iyi ağırlıklarla baştan tahmin al
    Ekle "Görselleştirme 2" -> Yeni tahminlerle Karmaşıklık Matrisi, ROC ve PR eğrilerini çizip Drive'a kaydet
    BİTİR